# LeetCode 1903 — Largest Odd Number in String bằng Python

Link đề: https://leetcode.com/problems/largest-odd-number-in-string/

Độ khó: **Easy** · Chủ đề: Math, String, Greedy

```python
class Solution:
    def largestOddNumber(self, num: str) -> str:

```

**Mục tiêu học:**

1. Hiểu **substring** (chuỗi con liên tiếp) và cách sinh mọi substring bằng hai vòng lặp + slicing `num[i:j]`.
2. Nắm hai sự thật toán học: số **lẻ** ⇔ **chữ số cuối** lẻ; với số không có số 0 ở đầu, **dài hơn ⇒ lớn hơn**.
3. Từ đó suy ra lời giải tham lam: tìm chữ số lẻ **xa nhất bên phải**, trả về tiền tố kết thúc tại đó.
4. Biết cách Pythonic `num.rstrip("02468")` và vì sao không nên `int(num)` với chuỗi dài 10^5.
5. Trace, chứng minh, so sánh Big-O giữa brute force `O(n³)` và tối ưu `O(n)`.

## 1. Bài toán yêu cầu gì?

Cho chuỗi `num` biểu diễn một số nguyên lớn (không có số 0 ở đầu). Trả về **số lẻ lớn nhất** (dạng chuỗi) là một **substring khác rỗng** của `num`, hoặc `""` nếu không có số lẻ nào.

**Substring** = dãy ký tự **liên tiếp** trong chuỗi.

| num | Kết quả | Vì sao |
|---|---|---|
| `"52"` | `"5"` | các substring: `"5"`, `"2"`, `"52"`; chỉ `"5"` lẻ |
| `"4206"` | `""` | không có chữ số lẻ nào |
| `"35427"` | `"35427"` | bản thân đã là số lẻ (tận cùng 7) |
| `"10"` | `"1"` | `"10"` chẵn, `"1"` lẻ |

**Ràng buộc:** `1 <= len(num) <= 10^5`, `num` chỉ gồm chữ số, không có số 0 ở đầu.

Chú ý: độ dài tới **100.000** chữ số — lớn hơn rất nhiều so với số `int` 64-bit (khoảng 19 chữ số).

In [1]:
examples = [
    ("52", "5"),
    ("4206", ""),
    ("35427", "35427"),
    ("10", "1"),
]
for num, expected in examples:
    print(f"num={num!r:9} -> kỳ vọng {expected!r}")

num='52'      -> kỳ vọng '5'
num='4206'    -> kỳ vọng ''
num='35427'   -> kỳ vọng '35427'
num='10'      -> kỳ vọng '1'


## 2. Nền tảng Python cần biết

### 2.1. Slicing `s[i:j]` — lấy substring

`s[i:j]` lấy các ký tự từ chỉ số `i` **đến trước** `j`. `s[:j]` là **tiền tố** (prefix) độ dài `j`.

In [2]:
num = "35427"
print(num[0:2])     # '35'
print(num[1:4])     # '542'
print(num[:3])      # tiền tố dài 3: '354'
print(num[:len(num)] == num)

35
542
354
True


### 2.2. Sinh mọi substring bằng hai vòng lặp

Chọn điểm bắt đầu `i` và điểm kết thúc (không gồm) `j > i`. Chuỗi dài `n` có `n(n+1)/2` substring.

In [3]:
s = "523"
subs = []
for i in range(len(s)):
    for j in range(i + 1, len(s) + 1):
        subs.append(s[i:j])
print(subs, len(subs), "==", len(s) * (len(s) + 1) // 2)

['5', '52', '523', '2', '23', '3'] 6 == 6


### 2.3. Số lẻ ⇔ chữ số cuối lẻ

Một số thập phân `...d` có dạng `10·k + d`. Vì `10·k` luôn chẵn, tính chẵn/lẻ chỉ phụ thuộc `d`. Vậy chỉ cần nhìn **chữ số cuối**: `1, 3, 5, 7, 9` lẻ.

`int(ch)` đổi một ký tự chữ số sang số; `int(ch) % 2 == 1` kiểm tra lẻ. Cách khác: `ch in "13579"`.

In [4]:
for x in [35427, 4206, 52, 10]:
    last = str(x)[-1]
    print(f"{x}: chữ số cuối {last} -> lẻ? {int(last) % 2 == 1} | kiểm bằng % trên cả số: {x % 2 == 1}")

print("7" in "13579", "4" in "13579")

35427: chữ số cuối 7 -> lẻ? True | kiểm bằng % trên cả số: True
4206: chữ số cuối 6 -> lẻ? False | kiểm bằng % trên cả số: False
52: chữ số cuối 2 -> lẻ? False | kiểm bằng % trên cả số: False
10: chữ số cuối 0 -> lẻ? False | kiểm bằng % trên cả số: False
True False


### 2.4. Duyệt ngược bằng `range(n - 1, -1, -1)`

`range(start, stop, step)` với `step = -1` đi lùi; `stop = -1` để vẫn lấy chỉ số 0.

In [5]:
s = "4206"
for i in range(len(s) - 1, -1, -1):
    print(i, s[i])

3 6
2 0
1 2
0 4


### 2.5. `str.rstrip(chars)` — xóa các ký tự thuộc `chars` ở **cuối**

`"420600".rstrip("0")` → `"4206"`. Truyền một chuỗi gồm nhiều ký tự: xóa mọi ký tự **thuộc tập đó** ở cuối, cho đến khi gặp ký tự khác.

In [6]:
print("420600".rstrip("0"))
print("35420".rstrip("02468"))   # xóa 0, 2, 4 ở cuối -> '35'
print("4206".rstrip("02468"))    # xóa hết -> ''
print("35427".rstrip("02468"))   # 7 không thuộc tập -> giữ nguyên

4206
35

35427


## 3. Tính bằng tay và hai quan sát then chốt

Lấy `num = "35420"`. Các substring lẻ là những substring **tận cùng bằng 3 hoặc 5**:

- Tận cùng tại `'3'` (chỉ số 0): chỉ `"3"`.
- Tận cùng tại `'5'` (chỉ số 1): `"5"`, `"35"`.

Lớn nhất là `"35"`.

**Quan sát 1:** số lẻ ⇔ chữ số cuối lẻ → substring lẻ phải **kết thúc** tại một chữ số lẻ.

**Quan sát 2:** cố định điểm kết thúc `j`, substring **bắt đầu từ 0** là dài nhất. Vì `num` không có số 0 ở đầu, số dài hơn thì lớn hơn → chọn `num[:j+1]`.

**Quan sát 3:** giữa các tiền tố `num[:j+1]`, cái nào `j` lớn hơn thì dài hơn → lớn hơn. Vậy chọn chữ số lẻ có `j` **lớn nhất** (xa nhất bên phải).

In [7]:
num = "35420"
odd_positions = [j for j, ch in enumerate(num) if ch in "13579"]
print("vị trí chữ số lẻ:", odd_positions)
for j in odd_positions:
    print(f"  kết thúc tại {j}: tiền tố {num[:j + 1]!r}")
print("chọn j lớn nhất ->", num[:odd_positions[-1] + 1])

vị trí chữ số lẻ: [0, 1]
  kết thúc tại 0: tiền tố '3'
  kết thúc tại 1: tiền tố '35'
chọn j lớn nhất -> 35


## 4. Cách 1 — Brute force: thử mọi substring

Làm đúng như đề: sinh mọi substring, giữ những cái lẻ, chọn cái lớn nhất.

So sánh "lớn nhất" thế nào? Với chuỗi ngắn, dùng `int(sub)` là được (lưu ý: substring như `"05"` có số 0 ở đầu, `int("05") = 5`).

In [8]:
def largest_odd_brute(num: str) -> str:
    best = ""
    best_val = -1
    n = len(num)
    for i in range(n):
        for j in range(i + 1, n + 1):
            sub = num[i:j]
            if int(sub[-1]) % 2 == 1:
                val = int(sub)
                if val > best_val:
                    best_val, best = val, sub
    return best

for num, expected in examples:
    print(f"{num!r:9} -> {largest_odd_brute(num)!r:9} | kỳ vọng {expected!r}")

'52'      -> '5'       | kỳ vọng '5'
'4206'    -> ''        | kỳ vọng ''
'35427'   -> '35427'   | kỳ vọng '35427'
'10'      -> '1'       | kỳ vọng '1'


**Độ phức tạp:** có `O(n²)` substring, mỗi lần cắt và `int()` tốn `O(n)` → `O(n³)`. Với `n = 10^5` đó là 10^15 phép — **không thể** chạy kịp.

Ngoài ra `int()` trên chuỗi rất dài còn gặp giới hạn của Python (xem ngay dưới đây).

### Bổ sung: giới hạn khi đổi chuỗi dài sang `int`

Từ Python 3.11, `int(s)` với `s` dài hơn **4300** chữ số mặc định báo `ValueError` (để chống tấn công từ chối dịch vụ). Bài này `num` dài tới 10^5, nên **không được** dựa vào `int(num)`.

In [9]:
long_num = "1" * 5000
try:
    v = int(long_num)
    print("Phiên bản Python này đổi được, số chữ số:", len(str(v)))
except ValueError as e:
    print("ValueError:", str(e)[:80], "...")

ValueError: Exceeds the limit (4300 digits) for integer string conversion: value has 5000 di ...


## 5. Cách 2 — Chỉ xét tiền tố (brute force đã cải tiến)

Từ quan sát 2: chỉ cần xét các **tiền tố** `num[:j+1]`. Với `j` từ `n-1` lùi về 0, tiền tố nào lẻ đầu tiên chính là đáp án (dài nhất). Ta vẫn cắt chuỗi mỗi lần nên tệ nhất `O(n²)`.

In [10]:
def largest_odd_prefixes(num: str) -> str:
    for j in range(len(num), 0, -1):
        prefix = num[:j]
        if int(prefix[-1]) % 2 == 1:
            return prefix
    return ""

for num, expected in examples:
    print(f"{num!r:9} -> {largest_odd_prefixes(num)!r:9} | kỳ vọng {expected!r}")

'52'      -> '5'       | kỳ vọng '5'
'4206'    -> ''        | kỳ vọng ''
'35427'   -> '35427'   | kỳ vọng '35427'
'10'      -> '1'       | kỳ vọng '1'


**Nút thắt còn lại:** ta cắt `num[:j]` **trước** khi biết nó lẻ. Nhưng chỉ cần nhìn `num[j-1]` là đủ biết! Chỉ cắt **một lần** khi đã tìm thấy.

## 6. Lời giải chính — Tham lam: tìm chữ số lẻ xa nhất bên phải

**Đoạn 1 — duyệt từ phải sang trái:** `for i in range(len(num) - 1, -1, -1)`.

**Đoạn 2 — gặp chữ số lẻ thì dừng:** `if int(num[i]) % 2 == 1: return num[:i + 1]`.

**Đoạn 3 — không gặp chữ số lẻ nào:** `return ""`.

In [11]:
def largest_odd_greedy(num: str) -> str:
    for i in range(len(num) - 1, -1, -1):
        if int(num[i]) % 2 == 1:
            return num[:i + 1]
    return ""

for num, expected in examples:
    print(f"{num!r:9} -> {largest_odd_greedy(num)!r:9} | kỳ vọng {expected!r}")

'52'      -> '5'       | kỳ vọng '5'
'4206'    -> ''        | kỳ vọng ''
'35427'   -> '35427'   | kỳ vọng '35427'
'10'      -> '1'       | kỳ vọng '1'


### Vì sao duyệt từ **phải**?

Ta muốn chữ số lẻ có chỉ số **lớn nhất**. Duyệt từ phải thì cái đầu tiên gặp là cái cần tìm → dừng ngay. Nếu duyệt từ trái, phải đi hết chuỗi và nhớ vị trí cuối cùng (vẫn `O(n)` nhưng luôn tốn trọn `n` bước).

In [12]:
def largest_odd_left_scan(num: str) -> str:
    last = -1
    for i, ch in enumerate(num):
        if ch in "13579":
            last = i
    return num[:last + 1]          # last = -1 -> num[:0] = ''

for num, expected in examples:
    print(f"{num!r:9} -> {largest_odd_left_scan(num)!r:9} | kỳ vọng {expected!r}")

'52'      -> '5'       | kỳ vọng '5'
'4206'    -> ''        | kỳ vọng ''
'35427'   -> '35427'   | kỳ vọng '35427'
'10'      -> '1'       | kỳ vọng '1'


Mẹo nhỏ trong `largest_odd_left_scan`: khi không có chữ số lẻ, `last = -1` và `num[:0]` là chuỗi rỗng — không cần `if` riêng.

## 7. TRACE từng bước

Ví dụ `num = "3542068"`:

| i | `num[i]` | lẻ? | Hành động |
|---:|:---:|:---:|---|
| 6 | 8 | không | sang trái |
| 5 | 6 | không | sang trái |
| 4 | 0 | không | sang trái |
| 3 | 2 | không | sang trái |
| 2 | 4 | không | sang trái |
| 1 | 5 | **có** | trả `num[:2] = "35"` |

Chữ số `3` ở chỉ số 0 cũng lẻ nhưng không cần xét — `"35"` đã dài hơn.

In [13]:
def trace_largest_odd(num: str) -> str:
    print(f"num = {num!r}")
    for i in range(len(num) - 1, -1, -1):
        d = int(num[i])
        if d % 2 == 1:
            print(f"  i={i}: num[i]='{num[i]}' LẺ -> return num[:{i + 1}] = {num[:i + 1]!r}")
            return num[:i + 1]
        print(f"  i={i}: num[i]='{num[i]}' chẵn -> sang trái")
    print("  Không có chữ số lẻ -> return ''")
    return ""

trace_largest_odd("3542068")
print()
trace_largest_odd("4206")
print()
trace_largest_odd("35427")

num = '3542068'
  i=6: num[i]='8' chẵn -> sang trái
  i=5: num[i]='6' chẵn -> sang trái
  i=4: num[i]='0' chẵn -> sang trái
  i=3: num[i]='2' chẵn -> sang trái
  i=2: num[i]='4' chẵn -> sang trái
  i=1: num[i]='5' LẺ -> return num[:2] = '35'

num = '4206'
  i=3: num[i]='6' chẵn -> sang trái
  i=2: num[i]='0' chẵn -> sang trái
  i=1: num[i]='2' chẵn -> sang trái
  i=0: num[i]='4' chẵn -> sang trái
  Không có chữ số lẻ -> return ''

num = '35427'
  i=4: num[i]='7' LẺ -> return num[:5] = '35427'


'35427'

## 8. Cách 4 — Một dòng: `rstrip("02468")`

Lời giải tham lam thực chất là "**bỏ các chữ số chẵn ở cuối**". Đó chính xác là `num.rstrip("02468")`:

- Nếu cuối có chữ số lẻ → không xóa gì.
- Nếu toàn chẵn → xóa hết → `""`.

In [14]:
def largest_odd_rstrip(num: str) -> str:
    return num.rstrip("02468")

for num, expected in examples:
    print(f"{num!r:9} -> {largest_odd_rstrip(num)!r:9} | kỳ vọng {expected!r}")

'52'      -> '5'       | kỳ vọng '5'
'4206'    -> ''        | kỳ vọng ''
'35427'   -> '35427'   | kỳ vọng '35427'
'10'      -> '1'       | kỳ vọng '1'


## 9. Code nộp LeetCode

Bản tham lam rõ ràng (khuyên dùng khi phỏng vấn):

In [15]:
class Solution:
    def largestOddNumber(self, num: str) -> str:
        for i in range(len(num) - 1, -1, -1):
            if int(num[i]) % 2 == 1:
                return num[:i + 1]
        return ""


sol = Solution()
for num, expected in examples:
    assert sol.largestOddNumber(num) == expected
print("Solution (tham lam) PASS")

Solution (tham lam) PASS


In [16]:
class SolutionRstrip:
    def largestOddNumber(self, num: str) -> str:
        return num.rstrip("02468")


for num, expected in examples:
    assert SolutionRstrip().largestOddNumber(num) == expected
print("SolutionRstrip PASS")

SolutionRstrip PASS


## 10. Vì sao đúng?

Gọi `j*` là chỉ số **lớn nhất** có `num[j*]` lẻ. Ta chứng minh `num[:j*+1]` là đáp án.

1. **Hợp lệ:** `num[:j*+1]` là substring, tận cùng bằng chữ số lẻ → là số lẻ.
2. **Lớn nhất:** lấy một substring lẻ bất kỳ `num[i:j+1]`. Nó tận cùng ở chữ số lẻ nên `j <= j*`. Độ dài của nó là `j - i + 1 <= j* + 1`.
   - Nếu ngắn hơn: `num[:j*+1]` bắt đầu bằng `num[0] ≠ '0'` (không có số 0 ở đầu), nên là số có `j*+1` chữ số **thật sự** — lớn hơn mọi số có ít chữ số hơn (kể cả khi `num[i:j+1]` có số 0 ở đầu, giá trị của nó còn nhỏ hơn nữa).
   - Nếu dài bằng: bắt buộc `i = 0` và `j = j*` → chính là `num[:j*+1]`.
3. **Không có chữ số lẻ** → không substring nào lẻ → `""`.

Thuật toán duyệt từ phải, dừng tại chữ số lẻ đầu tiên gặp — đó đúng là `j*`.

## 11. Độ phức tạp và bảng so sánh

`n = len(num)` (≤ 10^5).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|:---:|:---:|---|
| 1. Mọi substring | `O(n²)` substring × `int` | `O(n³)` | `O(n)` | chỉ để kiểm tra; còn vướng giới hạn `int` |
| 2. Mọi tiền tố | cắt rồi kiểm tra | `O(n²)` | `O(n)` | vẫn chậm với `n = 10^5` |
| 3. Tham lam từ phải | tìm chữ số lẻ cuối | `O(n)` | `O(n)` cho kết quả | dừng sớm, rõ ràng |
| 4. `rstrip("02468")` | xóa chẵn ở cuối | `O(n)` | `O(n)` cho kết quả | ngắn nhất |

Bộ nhớ `O(n)` ở cách 3, 4 là cho **chuỗi kết quả** (đề yêu cầu trả về chuỗi); ngoài kết quả, bộ nhớ phụ là `O(1)`.

## 12. Bộ test kiểm tra

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS")

fast = [largest_odd_greedy, largest_odd_left_scan, largest_odd_rstrip, largest_odd_prefixes]
tests = [
    ("ví dụ 1", "52", "5"),
    ("ví dụ 2", "4206", ""),
    ("ví dụ 3", "35427", "35427"),
    ("một chữ số lẻ", "7", "7"),
    ("một chữ số chẵn", "8", ""),
    ("số 0", "0", ""),
    ("tận cùng nhiều số 0", "1000000", "1"),
    ("lẻ ở giữa", "2468135024", "2468135"),
    ("toàn chẵn dài", "2" * 50, ""),
]
for name, num, e in tests:
    for f in fast + [largest_odd_brute]:
        assert f(num) == e, (name, f.__name__)
    check(name + " (5 cách)", largest_odd_greedy(num), e)

ví dụ 1 (5 cách)               PASS
ví dụ 2 (5 cách)               PASS
ví dụ 3 (5 cách)               PASS
một chữ số lẻ (5 cách)         PASS
một chữ số chẵn (5 cách)       PASS
số 0 (5 cách)                  PASS
tận cùng nhiều số 0 (5 cách)   PASS
lẻ ở giữa (5 cách)             PASS
toàn chẵn dài (5 cách)         PASS


In [18]:
import random

random.seed(1903)
mismatch = 0
for _ in range(2000):
    n_ = random.randint(1, 9)
    num = str(random.randint(1, 9)) + "".join(random.choice("0123456789") for _ in range(n_ - 1))
    expected = largest_odd_brute(num)          # oracle chậm
    if any(f(num) != expected for f in fast):
        mismatch += 1
print("Số ca lệch so với brute force:", mismatch)
assert mismatch == 0

Số ca lệch so với brute force: 0


In [19]:
import time

big = "1" + "2" * (10**5 - 1)        # chữ số lẻ duy nhất ở đầu -> phải quét hết
t0 = time.perf_counter()
res = largest_odd_greedy(big)
t1 = time.perf_counter()
print("kết quả:", res, "| thời gian tham lam: %.4f s" % (t1 - t0))
assert res == "1" and largest_odd_rstrip(big) == "1"

kết quả: 1 | thời gian tham lam: 0.0073 s


## 13. Lỗi người mới hay gặp

### Lỗi 1: Đổi cả chuỗi sang `int`

```python
x = int(num)          # SAI với num dài: ValueError (>4300 chữ số) hoặc rất chậm
while x % 2 == 0:
    x //= 10
return str(x) if x else ""
```

Ý tưởng đúng (bỏ chữ số chẵn ở cuối), nhưng thao tác trên số khổng lồ. Hãy làm việc trực tiếp trên **chuỗi**.

In [20]:
def risky_int_version(num):
    x = int(num)
    while x and x % 2 == 0:
        x //= 10
    return str(x) if x else ""

print("Chuỗi ngắn vẫn chạy:", risky_int_version("35420"))
try:
    risky_int_version("3" + "2" * 6000)
    print("Phiên bản Python này không giới hạn — nhưng vẫn chậm hơn nhiều")
except ValueError:
    print("Chuỗi dài: ValueError -> không nộp được cách này")

Chuỗi ngắn vẫn chạy: 35
Chuỗi dài: ValueError -> không nộp được cách này


### Lỗi 2: Trả về chữ số lẻ **lớn nhất** thay vì **số** lẻ lớn nhất

```python
return max(ch for ch in num if ch in "13579")   # SAI: '9' < '35427' về giá trị
```

Đề hỏi substring có giá trị lớn nhất, không phải chữ số lớn nhất.

In [21]:
def wrong_max_digit(num):
    odds = [ch for ch in num if ch in "13579"]
    return max(odds) if odds else ""

print("Sai :", wrong_max_digit("35427"), "| Đúng:", largest_odd_greedy("35427"))

Sai : 7 | Đúng: 35427


### Lỗi 3: Duyệt từ trái và dừng ở chữ số lẻ **đầu tiên**

```python
for i in range(len(num)):
    if int(num[i]) % 2: return num[:i + 1]   # SAI: lấy tiền tố ngắn nhất
```

In [22]:
def wrong_first_odd(num):
    for i in range(len(num)):
        if int(num[i]) % 2 == 1:
            return num[:i + 1]
    return ""

print("Sai :", wrong_first_odd("35427"), "| Đúng:", largest_odd_greedy("35427"))

Sai : 3 | Đúng: 35427


### Lỗi 4: Lệch một khi cắt

`num[:i]` **không** gồm `num[i]` — bỏ mất chữ số lẻ vừa tìm được. Phải là `num[:i + 1]`.

### Lỗi 5: `range(len(num) - 1, 0, -1)` bỏ qua chỉ số 0

Với `num = "3"` hoặc `"30"`, chữ số lẻ duy nhất ở chỉ số 0 sẽ bị bỏ sót.

In [23]:
def wrong_stop(num):
    for i in range(len(num) - 1, 0, -1):     # SAI: stop = 0
        if int(num[i]) % 2 == 1:
            return num[:i + 1]
    return ""

print("Sai :", repr(wrong_stop("30")), "| Đúng:", repr(largest_odd_greedy("30")))

Sai : '' | Đúng: '3'


## 14. Mẹo nhận dạng pattern và biến thể

- "Tìm substring/số **lớn nhất** thỏa tính chất phụ thuộc **chữ số cuối**" → nghĩ tới **tiền tố** + duyệt từ phải (tham lam).
- "Số rất dài dạng chuỗi" → **không** đổi sang `int`; xử lý từng ký tự.
- Tính chất chia hết theo chữ số cuối: chẵn/lẻ (`d % 2`), chia hết cho 5 (`d ∈ {0,5}`), cho 10 (`d = 0`). Chia hết cho 3, 9 phụ thuộc **tổng** chữ số — khác hẳn!

Bài liên quan: 2496, 1323 (Maximum 69 Number), 402 (Remove K Digits), 2264 (Largest 3-Same-Digit Number in String).

### Biến thể 1: số **chẵn** lớn nhất

Chỉ đổi tập chữ số: tìm chữ số chẵn xa nhất bên phải — `num.rstrip("13579")`.

In [24]:
def largest_even(num):
    return num.rstrip("13579")

print(largest_even("35427"), "|", largest_even("135"), "|", largest_even("1234"))

3542 |  | 1234


### Biến thể 2: substring lớn nhất chia hết cho 5

In [25]:
def largest_div5(num):
    for i in range(len(num) - 1, -1, -1):
        if num[i] in "05":
            return num[:i + 1]
    return ""

print(largest_div5("1253"), "|", largest_div5("123"), "|", largest_div5("7050"))

125 |  | 7050


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Dự đoán output

Không chạy code: kết quả với `"1"`, `"22"`, `"123456"`, `"9080706"`, `"5000"`? Ghi vào `ex1_answers` (list chuỗi).

In [26]:
ex1_answers = None   # TODO

### Bài 2 — Điền chỗ trống: lời giải tham lam

In [27]:
def ex2_largest_odd(num: str) -> str:
    for i in range(len(num) - 1, -1, -1):
        # TODO: nếu num[i] lẻ thì trả về tiền tố đúng
        pass
    return ""

# for num, e in examples: assert ex2_largest_odd(num) == e

### Bài 3 — Đếm số substring lẻ

Trả về **số lượng** substring (tính theo vị trí, không loại trùng) của `num` là số lẻ. Yêu cầu `O(n)`.

Ví dụ `"135"`: mọi 6 substring đều lẻ → 6. `"52"`: chỉ `"5"` → 1.

In [28]:
def count_odd_substrings(num: str) -> int:
    # TODO: substring kết thúc tại i lẻ có (i + 1) cách chọn điểm bắt đầu
    pass

# assert count_odd_substrings("135") == 6
# assert count_odd_substrings("52") == 1
# assert count_odd_substrings("2468") == 0

### Bài 4 — Số lẻ **nhỏ nhất** (dạng chuỗi, là substring)

Trả về giá trị của substring lẻ **nhỏ nhất**, viết dạng chuỗi không có số 0 ở đầu. Nếu không có substring lẻ, trả `""`. Hãy suy nghĩ: một substring lẻ nhiều chữ số có thể nhỏ hơn mọi chữ số lẻ không?

In [29]:
def smallest_odd(num: str) -> str:
    # TODO
    pass

# assert smallest_odd("35427") == "3"
# assert smallest_odd("4206") == ""
# assert smallest_odd("98765") == "5"

### Bài 5 — Tự viết lời giải duyệt từ trái (nhớ vị trí cuối)

In [30]:
def ex5_largest_odd(num: str) -> str:
    # TODO
    pass

# for num, e in examples: assert ex5_largest_odd(num) == e

### Bài 6 — Mini challenge: substring lớn nhất chia hết cho 3 là **tiền tố**?

Viết `largest_prefix_div3(num)`: trả về **tiền tố dài nhất** của `num` có tổng chữ số chia hết cho 3 (số chia hết cho 3 ⇔ tổng chữ số chia hết cho 3), hoặc `""`. Yêu cầu `O(n)`.

In [31]:
def largest_prefix_div3(num: str) -> str:
    # TODO: tính tổng tiền tố, nhớ vị trí cuối cùng có tổng % 3 == 0
    pass

# assert largest_prefix_div3("1234") == "123"
# assert largest_prefix_div3("12") == "12"
# assert largest_prefix_div3("4") == ""

## 16. Gợi ý 3 mức

**Bài 1.** (1) Tìm chữ số lẻ cuối cùng. (2) `"123456"` → chữ số lẻ cuối là 5. (3) `"5000"` → `"5"`.

**Bài 2.** (1) Số lẻ ⇔ chữ số lẻ. (2) `int(num[i]) % 2 == 1`. (3) `return num[:i + 1]`.

**Bài 3.** (1) Substring lẻ ⇔ kết thúc tại chữ số lẻ. (2) Kết thúc tại `i` thì bắt đầu ở `0..i` → `i + 1` cách. (3) `sum(i + 1 for i, ch in enumerate(num) if ch in "13579")`.

**Bài 4.** (1) Mọi substring lẻ có giá trị ≥ chữ số cuối của nó (vd `"01"` có giá trị 1, `"35"` là 35 ≥ 5). (2) Vậy đáp án là chữ số lẻ nhỏ nhất xuất hiện trong `num`. (3) `min` của các chữ số lẻ.

**Bài 5.** (1) Biến `last = -1`. (2) Cập nhật khi gặp chữ số lẻ. (3) `return num[:last + 1]`.

**Bài 6.** (1) Tổng tiền tố `s`. (2) Mỗi lần `s % 3 == 0` thì nhớ `best = i`. (3) `return num[:best + 1]` với `best = -1` ban đầu.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [32]:
ans_ex1 = ["1", "", "12345", "90807", "5"]


def ans_ex2_largest_odd(num):
    for i in range(len(num) - 1, -1, -1):
        if int(num[i]) % 2 == 1:
            return num[:i + 1]
    return ""


def ans_count_odd_substrings(num):
    return sum(i + 1 for i, ch in enumerate(num) if ch in "13579")


def ans_smallest_odd(num):
    odds = [ch for ch in num if ch in "13579"]
    return min(odds) if odds else ""


def ans_ex5_largest_odd(num):
    last = -1
    for i, ch in enumerate(num):
        if ch in "13579":
            last = i
    return num[:last + 1]


def ans_largest_prefix_div3(num):
    s, best = 0, -1
    for i, ch in enumerate(num):
        s += int(ch)
        if s % 3 == 0:
            best = i
    return num[:best + 1]


assert ans_ex1 == [largest_odd_greedy(x) for x in ["1", "22", "123456", "9080706", "5000"]]
for num, e in examples:
    assert ans_ex2_largest_odd(num) == e and ans_ex5_largest_odd(num) == e
assert ans_count_odd_substrings("135") == 6 and ans_count_odd_substrings("52") == 1
assert ans_count_odd_substrings("2468") == 0
assert ans_smallest_odd("35427") == "3" and ans_smallest_odd("4206") == "" and ans_smallest_odd("98765") == "5"
assert ans_largest_prefix_div3("1234") == "123" and ans_largest_prefix_div3("12") == "12"
assert ans_largest_prefix_div3("4") == ""

# đối chiếu bài 3 và bài 4 với brute force trên chuỗi ngắn
random.seed(42)
for _ in range(300):
    num = str(random.randint(1, 9)) + "".join(random.choice("0123456789") for _ in range(random.randint(0, 6)))
    subs = [num[i:j] for i in range(len(num)) for j in range(i + 1, len(num) + 1)]
    odd_subs = [x for x in subs if int(x) % 2 == 1]
    assert ans_count_odd_substrings(num) == len(odd_subs)
    expect_small = str(min(int(x) for x in odd_subs)) if odd_subs else ""
    assert ans_smallest_odd(num) == expect_small
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def largestOddNumber(self, num: str) -> str:
        for i in range(len(num) - 1, -1, -1):   # từ phải sang trái
            if int(num[i]) % 2 == 1:            # chữ số lẻ xa nhất bên phải
                return num[:i + 1]              # tiền tố dài nhất kết thúc tại đó
        return ""
        # hoặc: return num.rstrip("02468")
```

**Khung tư duy:** tính chất phụ thuộc chữ số cuối + "lớn nhất" + không có số 0 đầu → **tiền tố dài nhất** kết thúc tại vị trí hợp lệ xa nhất.

**Checklist trước khi submit:**

- [ ] Không gọi `int(num)` trên cả chuỗi dài?
- [ ] Duyệt ngược tới **chỉ số 0** (`stop = -1`)?
- [ ] Cắt `num[:i + 1]` (gồm cả `num[i]`)?
- [ ] Trả `""` khi không có chữ số lẻ?
- [ ] Tên method `largestOddNumber`, trả về `str`?